# 🚀 NanoChat: Convert & Upload to HuggingFace Hub

This notebook walks you through:
1. **Setting up the environment** - Installing dependencies
2. **Cloning the NanoChat repository**
3. **Downloading the checkpoint** from HuggingFace
4. **Converting to HuggingFace format** - Using the conversion script
5. **Testing the converted model** - Quick sanity check
6. **Uploading to HuggingFace Hub** - Using your HF token

---

## ⚙️ Prerequisites: Set Up Colab Secrets

Before running this notebook, add the following secrets in Google Colab:

### 🔑 Required Secrets

1. Click the 🔑 **key icon** in the left sidebar
2. Add these **3 secrets** (toggle "Notebook access" ON for each):

| Secret Name | Value | Example |
|-------------|-------|--------|
| `HF_TOKEN` | Your HuggingFace token with **Write** permission | `hf_xxxx...` |
| `HF_USERNAME` | Your HuggingFace username | `johndoe` |
| `REPO_NAME` | Name for your uploaded model repo | `nanochat-d34-sft-hf` |

### 📝 How to get your HF Token:
1. Go to [HuggingFace Settings > Access Tokens](https://huggingface.co/settings/tokens)
2. Create a new token with **Write** permission
3. Copy and paste it as the `HF_TOKEN` secret

---

## Step 1: Check GPU & Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Install Dependencies

We need the latest transformers with NanoChat support.

In [ ]:
# Install transformers from main branch (includes NanoChat support)
!pip install -q git+https://github.com/huggingface/transformers.git

# Install other required packages
!pip install -q huggingface_hub safetensors accelerate

print("\n✅ Dependencies installed!")

In [ ]:
# Verify transformers has NanoChat support
try:
    from transformers import NanoChatConfig, NanoChatForCausalLM
    print("✅ transformers with NanoChat support is installed!")
except ImportError as e:
    print("❌ NanoChat support not found in transformers.")
    print("Please restart runtime and run the pip install cell again.")
    raise e

## Step 3: Clone NanoChat Repository

In [ ]:
import os
import sys

# Clone the nanochat repository
REPO_URL = "https://github.com/karpathy/nanochat.git"
REPO_DIR = "/content/nanochat"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# Add the repo to Python path so we can import nanochat modules
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"\n✅ Working directory: {os.getcwd()}")
print(f"✅ Python path includes: {REPO_DIR}")

## Step 4: Set Environment Variables

In [ ]:
import os

# Set base directory for nanochat cache
NANOCHAT_BASE_DIR = "/content/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR

# HuggingFace repo to download from
HF_SOURCE_REPO = "pankajmathur/nanochat-d34-finetuned"
os.environ["HF_REPO"] = HF_SOURCE_REPO

# Output directory for converted model
OUTPUT_DIR = "/content/nanochat-d34-sft-hf"

# Directories for checkpoint and tokenizer
TOKENIZER_DIR = f"{NANOCHAT_BASE_DIR}/tokenizer"
CHECKPOINT_DIR = f"{NANOCHAT_BASE_DIR}/chatsft_checkpoints/d34"

# Create directories
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ NANOCHAT_BASE_DIR: {NANOCHAT_BASE_DIR}")
print(f"✅ Source HF repo: {HF_SOURCE_REPO}")
print(f"✅ Checkpoint dir: {CHECKPOINT_DIR}")
print(f"✅ Tokenizer dir: {TOKENIZER_DIR}")
print(f"✅ Output directory: {OUTPUT_DIR}")

## Step 5: Download the Model Checkpoint

Download the SFT checkpoint and tokenizer from HuggingFace.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
import os

repo_id = os.environ.get("HF_REPO", "pankajmathur/nanochat-d34-finetuned")
base_dir = os.environ.get("NANOCHAT_BASE_DIR", "/content/nanochat_cache")

print(f"📥 Downloading from: {repo_id}")
print(f"📁 Target directory: {base_dir}")
print()

# Get list of all files in the repo
print("Fetching file list from HuggingFace...")
all_files = list_repo_files(repo_id)

# Filter for tokenizer and SFT checkpoint files
files_to_download = []
for f in all_files:
    if f.startswith("tokenizer/") or f.startswith("chatsft_checkpoints/d34/"):
        files_to_download.append(f)

print(f"Found {len(files_to_download)} files to download:")
for f in files_to_download:
    print(f"  - {f}")
print()

# Download each file
for filepath in files_to_download:
    local_path = os.path.join(base_dir, filepath)

    if os.path.exists(local_path):
        print(f"⏭️  {filepath} already exists, skipping")
        continue

    print(f"Downloading {filepath}...")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=filepath,
        local_dir=base_dir,
        local_dir_use_symlinks=False
    )
    print(f"✅ Downloaded {filepath}")

print("\n✅ All files downloaded successfully!")

In [ ]:
# Verify downloaded files
import os

print("📁 Tokenizer files:")
!ls -la {TOKENIZER_DIR}

print("\n📁 Checkpoint files:")
!ls -la {CHECKPOINT_DIR}

## Step 6: Convert to HuggingFace Format

This is the main conversion step. We'll run the conversion directly in Python.

In [ ]:
import os
import sys
import gc
import json
from pathlib import Path
import torch

# Ensure we're in the repo directory and it's in path
REPO_DIR = "/content/nanochat"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"🔄 Converting checkpoint...")
print(f"   Input: {CHECKPOINT_DIR}")
print(f"   Tokenizer: {TOKENIZER_DIR}")
print(f"   Output: {OUTPUT_DIR}")
print()

In [ ]:
# Import the conversion functions from the scripts module
from transformers import NanoChatConfig, NanoChatForCausalLM

# ============================================================================
# Conversion functions (from scripts/convert_to_hf.py)
# ============================================================================

def infer_kv_heads(hidden_size: int, num_attention_heads: int, state_dict: dict) -> int:
    """Infer number of key-value heads from the checkpoint weights."""
    key_weight = state_dict.get("transformer.h.0.attn.c_k.weight")
    if key_weight is None:
        return num_attention_heads
    rows = key_weight.shape[0]
    head_dim = hidden_size // num_attention_heads
    if rows % head_dim != 0:
        return num_attention_heads
    inferred = rows // head_dim
    print(f"Inferred {inferred} key_value heads from checkpoint")
    return max(inferred, 1)


def convert_layer(old_prefix: str, new_prefix: str) -> dict:
    """Map nanochat layer keys to HuggingFace transformers layer keys."""
    return {
        f"{old_prefix}.attn.c_q.weight": f"{new_prefix}.self_attn.q_proj.weight",
        f"{old_prefix}.attn.c_k.weight": f"{new_prefix}.self_attn.k_proj.weight",
        f"{old_prefix}.attn.c_v.weight": f"{new_prefix}.self_attn.v_proj.weight",
        f"{old_prefix}.attn.c_proj.weight": f"{new_prefix}.self_attn.o_proj.weight",
        f"{old_prefix}.mlp.c_fc.weight": f"{new_prefix}.mlp.fc1.weight",
        f"{old_prefix}.mlp.c_proj.weight": f"{new_prefix}.mlp.fc2.weight",
    }


def load_config_from_checkpoint(input_path: Path):
    """
    Load config from either meta_*.json or config.json in the checkpoint directory.
    Returns a NanoChatConfig object.
    """
    # Try to find meta_*.json first (nanochat native format)
    meta_files = list(input_path.glob("meta_*.json"))

    if meta_files:
        meta_file = meta_files[0]
        print(f"Loading config from {meta_file.name}")
        with open(meta_file, "r") as f:
            meta_config = json.load(f)

        # Extract model config from meta file
        if "model_config" in meta_config:
            model_config = meta_config["model_config"]
        else:
            model_config = meta_config

        # Map nanochat config parameters to HuggingFace NanoChat config parameters
        config_kwargs = {
            "vocab_size": model_config.get("vocab_size", 50304),
            "hidden_size": model_config.get("n_embd", 768),
            "num_hidden_layers": model_config.get("n_layer", 12),
            "num_attention_heads": model_config.get("n_head", 6),
            "num_key_value_heads": model_config.get("n_kv_head"),
            "max_position_embeddings": model_config.get("sequence_len", 2048),
            "intermediate_size": model_config.get("intermediate_size", model_config.get("n_embd", 768) * 4),
        }

        # Try to load existing config.json for additional parameters
        config_file = input_path / "config.json"
        if config_file.exists():
            print("Loading additional config from config.json")
            with open(config_file, "r") as f:
                extra_config = json.load(f)

            # Add additional parameters from config.json
            for key in [
                "hidden_act", "attention_dropout", "rms_norm_eps", "initializer_range",
                "logits_soft_cap", "attention_bias", "intermediate_size",
                "bos_token_id", "eos_token_id", "pad_token_id",
            ]:
                if key in extra_config:
                    config_kwargs[key] = extra_config[key]
                elif key == "attention_bias" and "qkv_bias" in extra_config:
                    config_kwargs[key] = extra_config["qkv_bias"]

            if "rope_theta" in extra_config:
                config_kwargs["rope_theta"] = extra_config["rope_theta"]
            if "rope_parameters" in extra_config:
                config_kwargs["rope_parameters"] = extra_config["rope_parameters"]
            elif "rope_scaling" in extra_config and extra_config["rope_scaling"] is not None:
                config_kwargs["rope_parameters"] = extra_config["rope_scaling"]

        config = NanoChatConfig(**config_kwargs)
        return config
    else:
        # Fallback to loading from config.json if it exists
        config_file = input_path / "config.json"
        if config_file.exists():
            print("Loading config from config.json")
            config = NanoChatConfig.from_pretrained(input_path)
            if hasattr(config, "qkv_bias") and not hasattr(config, "attention_bias"):
                config.attention_bias = config.qkv_bias
            return config
        else:
            raise ValueError(f"No config file found in {input_path}. Expected meta_*.json or config.json")


def write_model(input_dir, output_dir, safe_serialization=True):
    """Convert NanoChat model from original checkpoint format to HuggingFace format."""
    print("Converting the model.")
    os.makedirs(output_dir, exist_ok=True)

    input_path = Path(input_dir)

    # Load config
    config = load_config_from_checkpoint(input_path)
    print(f"Loaded config hidden_size={config.hidden_size} num_layers={config.num_hidden_layers}")

    # Load checkpoint - try model_*.pt first, then pytorch_model.bin
    checkpoint_files = list(input_path.glob("model_*.pt"))
    if checkpoint_files:
        checkpoint_files.sort(key=lambda x: int(x.stem.split("_")[-1]))
        checkpoint_path = checkpoint_files[-1]
    else:
        checkpoint_path = input_path / "pytorch_model.bin"

    print(f"Fetching all parameters from the checkpoint at {checkpoint_path}...")
    old_state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

    # Handle torch.compile prefix if present
    old_state = {k.removeprefix("_orig_mod."): v for k, v in old_state.items()}

    # Original nanochat weights are in bfloat16
    for key in old_state:
        if old_state[key].dtype == torch.float32:
            old_state[key] = old_state[key].to(torch.bfloat16)

    # Infer key-value heads from checkpoint
    inferred_kv = infer_kv_heads(config.hidden_size, config.num_attention_heads, old_state)
    config.num_key_value_heads = inferred_kv
    if config.num_attention_heads % config.num_key_value_heads != 0:
        print(f"Adjusting num_attention_heads from {config.num_attention_heads} to {config.num_key_value_heads}")
        config.num_attention_heads = config.num_key_value_heads

    print("Converting model...")
    state_dict = {}
    rename_map = {}

    def assign(old_key, new_key, old_state, state_dict, rename_map):
        tensor = old_state.get(old_key)
        if tensor is None:
            return
        state_dict[new_key] = tensor.clone()
        rename_map[old_key] = new_key

    # Convert embeddings and head
    assign("transformer.wte.weight", "model.embed_tokens.weight", old_state, state_dict, rename_map)
    assign("lm_head.weight", "lm_head.weight", old_state, state_dict, rename_map)

    # Convert layers
    for layer_idx in range(config.num_hidden_layers):
        old_prefix = f"transformer.h.{layer_idx}"
        new_prefix = f"model.layers.{layer_idx}"
        mapping = convert_layer(old_prefix, new_prefix)
        for old_key, new_key in mapping.items():
            assign(old_key, new_key, old_state, state_dict, rename_map)

    missing = [key for key in old_state.keys() if key not in rename_map]
    if missing:
        print(f"Skipped {len(missing)} legacy entries that have no equivalent in the shared implementation:")
        for key in missing[:10]:
            print(f"  - {key}")
        if len(missing) > 10:
            print(f"  ... and {len(missing) - 10} more")

    del old_state
    gc.collect()

    # Update config
    config.dtype = torch.bfloat16
    config.tie_word_embeddings = False

    # Load the checkpoint into the model
    print("Loading the checkpoint in a NanoChat model.")
    with torch.device("meta"):
        model = NanoChatForCausalLM(config)
    model.load_state_dict(state_dict, strict=True, assign=True)
    print("Checkpoint loaded successfully.")

    if hasattr(model.config, "_name_or_path"):
        del model.config._name_or_path

    print("Saving the model.")
    model.save_pretrained(output_dir, safe_serialization=safe_serialization)
    del state_dict, model

    # Safety check: reload the converted model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print("Reloading the model to check if it's saved correctly.")
    try:
        NanoChatForCausalLM.from_pretrained(output_dir, torch_dtype=torch.bfloat16, device_map="auto")
    except Exception as e:
        print(f"Note: device_map='auto' failed ({type(e).__name__}), trying without device_map...")
        NanoChatForCausalLM.from_pretrained(output_dir, torch_dtype=torch.bfloat16)
    print("Model reloaded successfully.")

    return config


def write_tokenizer(input_dir, output_dir):
    """Convert and save the tokenizer."""
    input_path = Path(input_dir)

    tokenizer_dir = input_path / "tokenizer"
    if not tokenizer_dir.exists():
        tokenizer_dir = input_path

    tokenizer_pkl = tokenizer_dir / "tokenizer.pkl"
    if not tokenizer_pkl.exists():
        tokenizer_pkl = input_path / "tokenizer.pkl"

    if tokenizer_pkl.exists():
        try:
            import pickle
            from transformers.integrations.tiktoken import convert_tiktoken_to_fast

            print(f"Converting tokenizer from {tokenizer_pkl}")
            with open(tokenizer_pkl, "rb") as f:
                tok_pkl = pickle.load(f)
            convert_tiktoken_to_fast(tok_pkl, output_dir)
            print("Converted tokenizer.pkl to HuggingFace format")
            return True
        except Exception as e:
            print(f"Warning: Failed to convert tokenizer.pkl: {e}")
            for filename in ("tokenizer.json", "tokenizer_config.json"):
                src = tokenizer_dir / filename
                if src.exists():
                    (Path(output_dir) / filename).write_bytes(src.read_bytes())
    else:
        for filename in ("tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"):
            src = tokenizer_dir / filename
            if src.exists():
                (Path(output_dir) / filename).write_bytes(src.read_bytes())

    print("Tokenizer saved successfully.")
    return True

print("✅ Conversion functions loaded!")

In [ ]:
# Run the model conversion
print("="*60)
print("Starting Model Conversion")
print("="*60)

write_model(
    input_dir=CHECKPOINT_DIR,
    output_dir=OUTPUT_DIR,
    safe_serialization=True
)

print("\n" + "="*60)
print("Model conversion complete!")
print("="*60)

In [ ]:
# Convert the tokenizer
print("="*60)
print("Converting Tokenizer")
print("="*60)

write_tokenizer(
    input_dir=TOKENIZER_DIR,
    output_dir=OUTPUT_DIR
)

print("\n" + "="*60)
print("Tokenizer conversion complete!")
print("="*60)

In [ ]:
# List converted files
print("\n📁 Converted model files:")
!ls -la {OUTPUT_DIR}

## Step 7: Test the Converted Model

Quick test to verify the model works correctly.

In [ ]:
import torch
import gc

# Clear any previous model from memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Use specific NanoChat classes instead of Auto classes
from transformers import NanoChatForCausalLM, PreTrainedTokenizerFast

print("🔄 Loading converted model for testing...")

# Load the tokenizer using PreTrainedTokenizerFast directly
tokenizer = PreTrainedTokenizerFast.from_pretrained(OUTPUT_DIR)

# Load the model using NanoChatForCausalLM
model = NanoChatForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(f"✅ Model loaded successfully!")
print(f"   Model type: {type(model).__name__}")
print(f"   Device: {next(model.parameters()).device}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# Test generation
test_prompt = "Hello, who are you?"

print(f"\n💬 Test prompt: {test_prompt}")
print("-" * 50)

# Tokenize the input
inputs = tokenizer(test_prompt, return_tensors="pt")

# Only use input_ids (NanoChat doesn't need token_type_ids)
input_ids = inputs["input_ids"].to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"🤖 Response:\n{response}")

In [ ]:
# Free up GPU memory before uploading
del model
del tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()
print("✅ GPU memory cleared")

---

## Step 8: Upload to HuggingFace Hub

### ⚠️ Make sure you've set up all 3 secrets!

Click the 🔑 **key icon** in the left sidebar and add:
- `HF_TOKEN` - Your HuggingFace token (with write permission)
- `HF_USERNAME` - Your HuggingFace username
- `REPO_NAME` - Name for your model repo (e.g., `nanochat-d34-sft-hf`)

In [ ]:
# Get secrets from Colab
from google.colab import userdata

# Retrieve all required secrets
HF_TOKEN = None
HF_USERNAME = None
REPO_NAME = None

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN found!")
except:
    print("❌ HF_TOKEN not found. Please add it to Colab secrets.")

try:
    HF_USERNAME = userdata.get('HF_USERNAME')
    print(f"✅ HF_USERNAME found: {HF_USERNAME}")
except:
    print("❌ HF_USERNAME not found. Please add it to Colab secrets.")

try:
    REPO_NAME = userdata.get('REPO_NAME')
    print(f"✅ REPO_NAME found: {REPO_NAME}")
except:
    print("❌ REPO_NAME not found. Please add it to Colab secrets.")

# Check if all secrets are available
if HF_TOKEN and HF_USERNAME and REPO_NAME:
    TARGET_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
    print(f"\n📤 Target repository: https://huggingface.co/{TARGET_REPO_ID}")
else:
    print("\n⚠️  Missing secrets! Please add all 3 secrets to continue:")
    print("   1. Click the 🔑 key icon in the left sidebar")
    print("   2. Add: HF_TOKEN, HF_USERNAME, REPO_NAME")
    print("   3. Toggle 'Notebook access' ON for each")

In [ ]:
# Login to HuggingFace
from huggingface_hub import login, HfApi

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace Hub!")

    # Verify login by getting user info
    api = HfApi()
    user_info = api.whoami()
    print(f"   Logged in as: {user_info['name']}")
else:
    print("❌ Cannot login without HF_TOKEN. Please set up your token first.")

In [ ]:
# Create the repository (if it doesn't exist) and upload
from huggingface_hub import HfApi, create_repo

if HF_TOKEN and HF_USERNAME and REPO_NAME:
    api = HfApi()

    # Create the repository (will do nothing if it already exists)
    try:
        create_repo(
            repo_id=TARGET_REPO_ID,
            repo_type="model",
            exist_ok=True,
            private=False  # Set to True if you want a private repo
        )
        print(f"✅ Repository created/verified: {TARGET_REPO_ID}")
    except Exception as e:
        print(f"⚠️  Note: {e}")

    # Upload all files from the output directory
    print(f"\n📤 Uploading model to {TARGET_REPO_ID}...")
    print("   This may take a few minutes...\n")

    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=TARGET_REPO_ID,
        repo_type="model",
        commit_message="Upload NanoChat model converted to HuggingFace format"
    )

    print(f"\n✅ Upload complete!")
    print(f"🔗 View your model at: https://huggingface.co/{TARGET_REPO_ID}")
else:
    print("❌ Cannot upload. Please set up all required secrets (HF_TOKEN, HF_USERNAME, REPO_NAME).")

## Step 9: Add a Model Card (Optional but Recommended)

Create a nice README for your model on HuggingFace Hub.

In [ ]:
# Create a model card
if HF_USERNAME and REPO_NAME:
    TARGET_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

    MODEL_CARD = f"""---
license: apache-2.0
language:
- en
library_name: transformers
pipeline_tag: text-generation
tags:
- nanochat
- conversational
- pytorch
---

# NanoChat D34 SFT - HuggingFace Format

This is the NanoChat D34 model converted to HuggingFace Transformers format.

## Model Description

NanoChat is a lightweight conversational AI model designed for efficient inference.

- **Model type:** Causal Language Model
- **Language:** English
- **License:** Apache 2.0

## Usage

```python
from transformers import NanoChatForCausalLM, PreTrainedTokenizerFast
import torch

# Load model and tokenizer
model = NanoChatForCausalLM.from_pretrained(
    "{TARGET_REPO_ID}",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = PreTrainedTokenizerFast.from_pretrained("{TARGET_REPO_ID}")

# Generate text
prompt = "Hello, who are you?"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(model.device)

outputs = model.generate(
    input_ids,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Source

- Original checkpoint: [pankajmathur/nanochat-d34-finetuned](https://huggingface.co/pankajmathur/nanochat-d34-finetuned)
- Repository: [nanochat](https://github.com/karpathy/nanochat)

## Citation

If you use this model, please cite the original NanoChat project.
"""

    # Save model card locally
    readme_path = f"{OUTPUT_DIR}/README.md"
    with open(readme_path, "w") as f:
        f.write(MODEL_CARD)

    print(f"✅ Model card saved to {readme_path}")
    print("\nPreview:")
    print("-" * 50)
    print(MODEL_CARD[:500] + "...")
else:
    print("⚠️  Please set HF_USERNAME and REPO_NAME secrets first.")

In [ ]:
# Upload the model card
from huggingface_hub import HfApi

if HF_TOKEN and HF_USERNAME and REPO_NAME:
    api = HfApi()

    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=TARGET_REPO_ID,
        repo_type="model",
        commit_message="Add model card"
    )

    print(f"✅ Model card uploaded!")
    print(f"🔗 View at: https://huggingface.co/{TARGET_REPO_ID}")
else:
    print("⚠️  Skipping upload - please set all secrets first.")

---

## ✅ All Done!

Your NanoChat model has been:
1. Downloaded from HuggingFace
2. Converted to HuggingFace Transformers format
3. Tested locally
4. Uploaded to your HuggingFace Hub repository

### Next Steps

- Visit your model page on HuggingFace Hub
- Try the model using the Inference API
- Share with the community!

### Troubleshooting

**If you see "NanoChat not found" errors:**
- Make sure you installed transformers from the main branch
- Restart the runtime and run all cells again

**If upload fails:**
- Check that your HF_TOKEN has write permission
- Verify all secrets are correctly added to Colab

In [ ]:
# Final summary
print("=" * 60)
print("📋 SUMMARY")
print("=" * 60)
print(f"\n✅ Source checkpoint: {HF_SOURCE_REPO}")
print(f"✅ Converted model: {OUTPUT_DIR}")
if HF_USERNAME and REPO_NAME:
    print(f"✅ Uploaded to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")
else:
    print(f"⚠️  Not uploaded (set HF_USERNAME and REPO_NAME secrets to upload)")
print("\n" + "=" * 60)